In [0]:
%run ../silver/00_silver_helpers

In [0]:
df= read_table('customers')
display(df)

In [0]:
df.printSchema()

In [0]:
logger = get_logger("silver_customers")

try:

    logger.info("Starting Silver Customers transformation")


    # ============================================================
    # 1. Data type conversion and string trimming
    # ============================================================

    logger.info("Updating data types of the columns")

    logger.info(
        "Trimming leading/trailing spaces from string columns"
    )

    df = df.select(
        trim(col("customer_id"))
            .cast("string")
            .alias("customer_id"),

        trim(col("customer_name"))
            .cast("string")
            .alias("customer_name"),

        trim(col("email"))
            .cast("string")
            .alias("email"),

        trim(col("city"))
            .cast("string")
            .alias("city"),

        trim(col("state"))
            .cast("string")
            .alias("state"),

        col("registration_date")
            .cast("date")
            .alias("registration_date"),

        col("_ingestion_timestamp"),

        col("_source_file")
    )

    logger.info(
        "Data type conversion and string trimming completed"
    )


    # ============================================================
    # 2. Remove duplicate customers
    # ============================================================

    logger.info("Checking for duplicate customer_id values")

    before_count = df.count()

    distinct_count = df.dropDuplicates(
        ["customer_id"]
    ).count()

    duplicate_count = before_count - distinct_count

    logger.info(
        f"Duplicate customer_id records found: {duplicate_count}"
    )

    df = df.dropDuplicates(
        ["customer_id"]
    )

    logger.info(
        "Duplicate customer_id records removed"
    )


    # ============================================================
    # 3. Print schema
    # ============================================================

    logger.info("Displaying Silver Customers schema")

    df.printSchema()

    display(df)


    # ============================================================
    # 4. Create schema
    # ============================================================

    logger.info(
        f"Creating schema if it does not exist: "
        f"{catalog_name}.{schema_name}"
    )

    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS
        {catalog_name}.{schema_name}
        """
    )

    logger.info(
        f"Schema ready: {catalog_name}.{schema_name}"
    )


    # ============================================================
    # 5. Save Silver Customers
    # ============================================================

    logger.info("Saving Silver Customers table")

    save_table(
        df,
        "customers_clean"
    )

    logger.info(
        "Silver Customers table saved successfully"
    )

    logger.info(
        "Silver Customers transformation completed successfully"
    )


except Exception:
    
    logger.exception(
        "Silver Customers transformation failed"
    )

    raise